# Bonus · Multi-Region & Sharding (Leave-Behind)

Not part of any live demo module — this is for AMEX engineers to run after
the workshop. It complements the deck's existing "How to Set Up a Sharded
Cluster" section (which is UI screenshots only) with something runnable,
plus a short look at multi-region write-region priority.

## What's here

1. **Inspect your current cluster topology** via the Atlas Admin API —
   regions, shard count, electable nodes per region.
2. **Shard distribution check** — for an already-sharded cluster/collection,
   query chunk distribution across shards so you can see how evenly data is
   spread.
3. **Multi-region notes** — how region priority affects election order and
   what to check before promoting a non-primary-region node.

Fill in `ATLAS_PUBLIC_KEY` / `ATLAS_PRIVATE_KEY` / `ATLAS_PROJECT_ID` /
`ATLAS_CLUSTER_NAME` in `.env` (or when prompted in Colab) to use the Admin
API cells below.


In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pymongo[encryption]", "certifi", "python-dotenv", "requests", "matplotlib", "pandas"],
        check=True,
    )
    from getpass import getpass
    ATLAS_URI = os.environ.get("ATLAS_URI") or getpass("Atlas connection string (ATLAS_URI): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    ATLAS_URI = os.environ["ATLAS_URI"]

DEMO_DB = os.environ.get("DEMO_DB", "amex_demo")
import certifi
CA_FILE = certifi.where()
print("Environment:", "Colab" if IN_COLAB else "local", "| DB:", DEMO_DB)

from requests.auth import HTTPDigestAuth
import requests

PUBLIC_KEY = os.environ.get("ATLAS_PUBLIC_KEY")
PRIVATE_KEY = os.environ.get("ATLAS_PRIVATE_KEY")
PROJECT_ID = os.environ.get("ATLAS_PROJECT_ID")
CLUSTER_NAME = os.environ.get("ATLAS_CLUSTER_NAME")

BASE = "https://cloud.mongodb.com/api/atlas/v2"
HEADERS = {"Accept": "application/vnd.atlas.2023-11-15+json"}

def atlas_get(path):
    resp = requests.get(
        f"{BASE}{path}",
        auth=HTTPDigestAuth(PUBLIC_KEY, PRIVATE_KEY),
        headers=HEADERS,
    )
    resp.raise_for_status()
    return resp.json()


In [ ]:
# Requires ATLAS_PUBLIC_KEY / ATLAS_PRIVATE_KEY / ATLAS_PROJECT_ID / ATLAS_CLUSTER_NAME.
cluster = atlas_get(f"/groups/{PROJECT_ID}/clusters/{CLUSTER_NAME}")
for spec in cluster.get("replicationSpecs", []):
    for region in spec.get("regionConfigs", []):
        print(region.get("regionName"), "- electable nodes:",
              region.get("electableSpecs", {}).get("nodeCount"))


In [ ]:
from pymongo import MongoClient

client = MongoClient(ATLAS_URI, tlsCAFile=CA_FILE)

# If the target database is sharded, this shows chunk distribution per shard.
try:
    stats = client.admin.command("listShards")
    print("Shards:", [s["_id"] for s in stats.get("shards", [])])
except Exception as e:
    print("Not a sharded cluster, or insufficient privileges:", e)


### Presenter/leave-behind notes

- Region priority in a multi-region replica set determines which region's
  nodes are preferred for the primary — reorder it in the Atlas UI under
  cluster configuration if a failover should prefer staying in-region.
- Sharding in Atlas is a config change, not a migration project — walk
  through **Cluster → Configuration → Sharding** to see the same flow shown
  statically in the deck, live, against this cluster.
